# FX Forward Pricing Demo

This notebook demonstrates how to price an FX Forward using the `atlas` library.

We show two approaches:
1. **Pricing Dispatcher**: Constructing a `MarketDataSnapshot` and an `FxForward` instrument and using the pricing dispatcher (`calculate_price`).
2. **Direct FX Forward Pricer**: Directly using the FX Forward pricing function (`fx_forward_pricer`).

## Method 1: Using the Pricing Dispatcher

In this method, we construct standard domain objects: `FxForward` and `MarketDataSnapshot`, and call the central dispatcher function `calculate_price`.

In [1]:
import QuantLib as ql
from atlas.domain.instruments.fx.fx_forward import FxForward
from atlas.domain.market.market_data import (
    MarketDataSnapshot,
    FXSpot,
    FixedRate,
)
from atlas.domain.enums import Currency
from atlas.compute.pricing.pricing_dispatcher import calculate_price

# 1. Setup Dates
valuation_date = ql.Date(22, 6, 2026)
settlement_date = ql.Date(22, 6, 2027) # 1 year to settlement

# 2. Construct the FX Forward Instrument
fx_forward = FxForward(
    base_ccy=Currency.EUR,
    quote_ccy=Currency.USD,
    strike_forward_rate=1.10,
    notional=1_000_000.0,  # 1M EUR
    settlement_date=settlement_date,
)

# 3. Construct the Market Data Snapshot
market_data = MarketDataSnapshot(
    valuation_date=valuation_date,
    equity_spots={},
    fixed_rate={
        Currency.EUR: FixedRate(rate=0.03),  # EUR risk-free interest rate (domestic)
        Currency.USD: FixedRate(rate=0.04),  # USD risk-free interest rate (foreign)
    },
    volatility_rates={},
    dividend_rates={},
    fx_spots={
        (Currency.EUR, Currency.USD): FXSpot(
            base_ccy=Currency.EUR,
            quote_ccy=Currency.USD,
            value=1.12,
        )
    },
)

# 4. Price the FX Forward using the Dispatcher
price_dispatcher = calculate_price(fx_forward, market_data)
print(f"FX Forward Price (via Dispatcher): {price_dispatcher:.6f}")

FX Forward Price (via Dispatcher): 8594.084947


## Method 2: Direct FX Forward Pricing

In this method, we directly call the low-level `fx_forward_pricer` with the exact same inputs as the instrument above to verify the pricing result.

In [2]:
from atlas.compute.pricing.fx.fx_forward_pricer import fx_forward_pricer

# Price the FX Forward directly
price_direct = fx_forward_pricer(
    valuation_date=valuation_date,
    settlement_date=settlement_date,
    strike_forward_rate=1.10,
    notional=1_000_000.0,
    spot_exchange_rate=1.12,
    foreign_interest_rate=0.04,
    domestic_interest_rate=0.03,
)
print(f"FX Forward Price (Direct BSM):      {price_direct:.6f}")
print(f"Verification (Prices Match):         {price_dispatcher == price_direct}")

FX Forward Price (Direct BSM):      8594.084947
Verification (Prices Match):         True
